# 04: Training CNN-STFT Model

This notebook demonstrates training a CNN-STFT model for supervised scar tissue classification.

## Workflow:
1. Load and prepare data
2. Generate multi-scale STFT spectrograms  
3. Create CNN-STFT model
4. Train model with early stopping
5. Evaluate on test set

## Notes
- This notebook uses TensorFlow/Keras for implementation
- Install: `pip install tensorflow>=2.10.0`

In [ ]:
# Setup
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Import libraries
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("Warning: TensorFlow not installed. Install with: pip install tensorflow>=2.10.0")
    print("Continuing with model specification only (no actual training).")

from src import config, data_loading, data_processing, utils
from src.models import cnn_stft

In [ ]:
# Load configuration
cfg = config.get_config("config.yaml")
cfg.ensure_directories_exist()

print("CNN-STFT Configuration:")
for key, value in cfg.model_params['cnn_stft'].items():
    print(f"  {key}: {value}")

## 1. Load and Prepare Data

In [ ]:
# Generate synthetic training data
print("Generating training data...")
n_train = 100
n_channels = 3
n_timesteps = 2500

X_train_raw = data_processing.generate_synthetic_egm_data(
    n_samples=n_train, n_channels=n_channels, n_timesteps=n_timesteps, random_state=42
)
y_train = data_processing.generate_synthetic_labels(n_samples=n_train, random_state=42)

n_val = 25
n_test = 25

X_val_raw = data_processing.generate_synthetic_egm_data(
    n_samples=n_val, n_channels=n_channels, n_timesteps=n_timesteps, random_state=43
)
y_val = data_processing.generate_synthetic_labels(n_samples=n_val, random_state=43)

X_test_raw = data_processing.generate_synthetic_egm_data(
    n_samples=n_test, n_channels=n_channels, n_timesteps=n_timesteps, random_state=44
)
y_test = data_processing.generate_synthetic_labels(n_samples=n_test, random_state=44)

print(f"Train: {X_train_raw.shape}, Val: {X_val_raw.shape}, Test: {X_test_raw.shape}")

In [ ]:
# Preprocess signals
print("Filtering signals...")
X_train_filt = data_processing.bandpass_filter(X_train_raw)
X_val_filt = data_processing.bandpass_filter(X_val_raw)
X_test_filt = data_processing.bandpass_filter(X_test_raw)

print("Normalizing channels...")
X_train_norm = data_processing.normalize_channels(X_train_filt)
X_val_norm = data_processing.normalize_channels(X_val_filt)
X_test_norm = data_processing.normalize_channels(X_test_filt)

print(f"Preprocessed ranges:")
print(f"  Train: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]")

## 2. Generate Multi-Scale Spectrograms

In [ ]:
# Generate spectrograms
print("Computing multi-scale STFT spectrograms...")
print("  This may take a minute for large datasets...")

X_train_spec = data_processing.compute_multiscale_spectrograms(
    X_train_norm,
    frame_lengths=[64, 256, 512],
    frame_steps=[32, 128, 256],
    output_size=128
)

X_val_spec = data_processing.compute_multiscale_spectrograms(
    X_val_norm,
    frame_lengths=[64, 256, 512],
    frame_steps=[32, 128, 256],    
    output_size=128
)

X_test_spec = data_processing.compute_multiscale_spectrograms(
    X_test_norm,
    frame_lengths=[64, 256, 512],
    frame_steps=[32, 128, 256],
    output_size=128
)

print(f"\nSpectrogram shapes:")
print(f"  Train: {X_train_spec.shape}")
print(f"  Val: {X_val_spec.shape}")
print(f"  Test: {X_test_spec.shape}")

utils.print_batch_info(X_train_spec, y_train, "Training Spectrograms")

## 3. Create CNN-STFT Model Specification

In [ ]:
# Get model specification
model_spec = cnn_stft.create_cnn_stft_model(
    input_shape=(128, 128, 9),  # (height, width, channels)
    n_classes=3,  # Endocardial, mid-myocardial, epicardial
    initial_filters=16,
    dropout_rate=0.2,
    l2_regularization=1e-4
)

print("\nCNN-STFT Model Specification:")
print(f"  Input shape: {model_spec['input_shape']}")
print(f"  Architecture: {model_spec['architecture']['type']}")
print(f"  Backbone: {model_spec['architecture']['backbone']}")
print(f"  Number of layers: {len(model_spec['layers'])}")

In [ ]:
# Compute class weights for imbalanced data
class_weights = data_processing.compute_class_weights(y_train)
print(f"\nClass weights (for loss weighting):")
label_names = cfg.get_data_param('label_names')
for i, name in enumerate(label_names):
    print(f"  {name}: {class_weights[i]:.4f}")

## 4. Build and Train Model (TensorFlow)

In [ ]:
# Note: This section requires TensorFlow to be installed
if 'keras' not in dir():
    print("Skipping TensorFlow training - TensorFlow not installed")
    print("For actual training, install: pip install tensorflow>=2.10.0")
else:
    print("Building model from specification...")
    
    # This would be the actual Keras implementation
    # For demonstration, we just show the specification
    print("Model configuration saved.")
    
    # Save configuration
    model_config_path = cfg.paths['output_dir'] / "cnn_stft_model_config.json"
    utils.save_model_config(model_spec, str(model_config_path))

In [ ]:
# Example of how to build and train in TensorFlow
print("""
# ===== EXAMPLE: How to train the model in TensorFlow =====

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Build model
model = keras.Sequential([
    layers.Input(shape=(128, 128, 9)),
    
    # Initial conv block
    layers.Conv2D(16, 3, padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    
    # Residual blocks (3 scales with pooling)
    layers.Conv2D(32, 3, padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.2),
    
    layers.Conv2D(64, 3, padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.2),
    
    layers.Conv2D(128, 3, padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.2),
    
    # Global pooling and dense layers
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    
    # Output
    layers.Dense(3, activation="sigmoid")  # Multi-label
])

# Compile with weighted loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['binary_accuracy', tf.keras.metrics.AUC()]
)

# Train with class weights
history = model.fit(
    X_train_spec, y_train,
    validation_data=(X_val_spec, y_val),
    epochs=10,
    batch_size=32,
    class_weight={0: class_weights[0], 1: class_weights[1], 2: class_weights[2]},
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test_spec, y_test)
print(f"Test accuracy: {test_acc:.4f}")
""")

## 5. Model Hyperparameter Tuning Options

In [ ]:
# Available hyperparameters for tuning
print("Available hyperparameters for tuning:")
for param, values in cnn_stft.CNNSTFTHyperModel.HYPERPARAMETER_SPACE.items():
    print(f"\n  {param}:")
    print(f"    Options: {values}")

## Summary

In this notebook, we:
1. Loaded and preprocessed EGM signals
2. Generated multi-scale STFT spectrograms
3. Created CNN-STFT model specification
4. Demonstrated training pipeline
5. Showed hyperparameter tuning options

**To actually train the model**:
1. Install TensorFlow: `pip install tensorflow>=2.10.0`
2. Uncomment and run the TensorFlow training code in section 4
3. Evaluate results in notebook `06_evaluate_models.ipynb`

**Configuration**: Edit `config.yaml` to change:
- `batch_size`, `epochs`, `learning_rate`
- `initial_filters`, `dropout_rate`, `l2_regularization`
- `stft_scales`, `spectrogram_size`

In [ ]:
print("Training preparation complete! Model is ready for training.")